# Step 1: Load Source Data
Load the two raw CSV files and confirm the dataset shape before any transformations.

In [6]:
raw_data_path <- "data/raw/diabetic_data.csv"
ids_map_path <- "data/raw/IDS_mapping.csv"

if (!file.exists(raw_data_path) || !file.exists(ids_map_path)) {
  stop("Missing CSV input files. Expected data/raw/diabetic_data.csv and data/raw/IDS_mapping.csv")
}

diabetic <- read.csv(
  raw_data_path,
  stringsAsFactors = FALSE,
  na.strings = c("", "?", "NULL")
)

ids_raw <- read.csv(
  ids_map_path,
  header = FALSE,
  stringsAsFactors = FALSE,
  fill = TRUE,
  na.strings = c("", "NULL")
)

colnames(ids_raw) <- c("key", "description")

cat("diabetic_data.csv shape:", sprintf("(%d, %d)", nrow(diabetic), ncol(diabetic)), "\n")
cat("IDS_mapping raw shape:", sprintf("(%d, %d)", nrow(ids_raw), ncol(ids_raw)), "\n\n")

cat("diabetic_data preview:\n")
print(head(diabetic[, c("encounter_id", "patient_nbr", "age", "readmitted")]))

diabetic_data.csv shape: (101766, 50) 
IDS_mapping raw shape: (68, 2) 

diabetic_data preview:
  encounter_id patient_nbr     age readmitted
1      2278392     8222157  [0-10)         NO
2       149190    55629189 [10-20)        >30
3        64410    86047875 [20-30)         NO
4       500364    82442376 [30-40)         NO
5        16680    42519267 [40-50)         NO
6        35754    82637451 [50-60)        >30


# Step 2: Parse IDS Mapping Tables
Split IDS_mapping.csv into three mapping tables: admission type, discharge disposition, and admission source.

In [8]:
header_rows <- which(grepl("_id$", ids_raw$key) & ids_raw$description == "description")

extract_mapping <- function(section_name) {
  start_idx <- which(ids_raw$key == section_name & ids_raw$description == "description")
  if (length(start_idx) == 0) return(data.frame(id = integer(), description = character()))
  start_idx <- start_idx[1]

  next_headers <- header_rows[header_rows > start_idx]
  end_idx <- if (length(next_headers) > 0) next_headers[1] - 1 else nrow(ids_raw)

  out <- ids_raw[(start_idx + 1):end_idx, , drop = FALSE]
  out <- out[!is.na(out$key) & nzchar(trimws(out$key)), , drop = FALSE]
  out$id <- suppressWarnings(as.integer(out$key))
  out <- out[!is.na(out$id), c("id", "description"), drop = FALSE]
  rownames(out) <- NULL
  out
}

admission_type_map <- extract_mapping("admission_type_id")
discharge_disposition_map <- extract_mapping("discharge_disposition_id")
admission_source_map <- extract_mapping("admission_source_id")

cat("admission_type_map preview:\n")
print(head(admission_type_map, 5))

cat("\ndischarge_disposition_map preview:\n")
print(head(discharge_disposition_map, 5))

cat("\nadmission_source_map preview:\n")
print(head(admission_source_map, 5))

admission_type_map preview:
  id   description
1  1     Emergency
2  2        Urgent
3  3      Elective
4  4       Newborn
5  5 Not Available

discharge_disposition_map preview:
  id                                                          description
1  1                                                   Discharged to home
2  2                Discharged/transferred to another short term hospital
3  3                                        Discharged/transferred to SNF
4  4                                        Discharged/transferred to ICF
5  5 Discharged/transferred to another type of inpatient care institution

admission_source_map preview:
  id                                     description
1  1                              Physician Referral
2  2                                 Clinic Referral
3  3                                    HMO Referral
4  4                        Transfer from a hospital
5  5  Transfer from a Skilled Nursing Facility (SNF)
